In [ ]:
import kagglehub
path = kagglehub.dataset_download("jameslko/gun-violence-data")


100%|██████████| 33.6M/33.6M [00:00<00:00, 135MB/s]

Extracting files...


In [ ]:
import pandas as pd
data=pd.read_csv("/kaggle/input/gun-violence-data/gun-violence-data_01-2013_03-2018.csv")
data

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/gun-violence-data/gun-violence-data_01-2013_03-2018.csv'

In [ ]:
import missingno as mo
mo.bar(data, labels=True)

In [ ]:
mo.matrix(data)

In [ ]:
data.drop(columns=['incident_url', 'source_url', 'incident_url_fields_missing', 'congressional_district', 'gun_stolen', 'location_description', 'n_guns_involved'], inplace=True)
data.drop(columns=['notes', 'participant_age', 'participant_name', 'participant_relationship','sources','state_house_district','state_senate_district'], inplace=True)

In [ ]:
data.insert(data.columns.get_loc('n_injured') + 1, 'total_victims', data['n_killed'] + data['n_injured'])

In [ ]:
data['date'] = pd.to_datetime(data['date'])
data.insert(data.columns.get_loc('date') + 1, 'year', data['date'].dt.year)
data.insert(data.columns.get_loc('date') + 2, 'month', data['date'].dt.month)
data.insert(data.columns.get_loc('date') + 3, 'days', data['date'].dt.day)
data.insert(data.columns.get_loc('date') + 4, 'day_of_week', data['date'].dt.day_of_week)
data['day_of_week'] = data['day_of_week'].map({0: 'Monday', 1: 'Tuesday',
                                               2: 'Wednesday', 3: 'Thursday',
                                               4: 'Friday', 5: 'Saturday', 6: 'Sunday'})


In [ ]:
data['incident_characteristics'] = data['incident_characteristics'].apply(lambda x : str(x).split('||'))
data['participant_gender'] = data['participant_gender'].apply(lambda x : dict(Male = str(x).count('Male'), Female = str(x).count('Female')))
def first_gun(s):
    if not isinstance(s, str):
        return None
    x = s.split("||")[0]
    result = x.split("::")
    return result[1] if len(result) > 1 else None

data['gun_type'] = data['gun_type'].apply(first_gun)


In [ ]:
most_freq_gun = data['gun_type'].mode()[0]
data['gun_type'] = data['gun_type'].fillna(most_freq_gun)

In [ ]:
data = data[data['total_victims'] != 0]

In [ ]:
data

In [ ]:
mo.bar(data)

In [ ]:
mo.matrix(data)

In [ ]:
#we will analyze which state have the most number of victims
c = data.groupby('state')['total_victims'].sum().sort_values(ascending=False)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(18, 8))
plt.bar(c.index, c.values, color='red')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.xlabel("States")
plt.ylabel("Total victims")
plt.title("Total Victims in each State")
plt.show()


In [ ]:
#Now we will calculate the total of victims in each year
x=data.groupby('year')['total_victims'].sum().sort_values(ascending=False)

In [ ]:
plt.bar(x.index, x.values, color='green')
plt.xlabel("year")
plt.ylabel("Total victims")
plt.title("Total Victims in each Year")
plt.show()


In [ ]:
#Now i want to calculate the number of male and female victims
Male_victims = data['participant_gender'].apply(lambda x : x.get('Male', 0)).sum()
Female_victims = data['participant_gender'].apply(lambda x : x.get('Female', 0)).sum()
print(f'number of Male victims: {Male_victims}')
print(f'number of Female victims: {Female_victims}')
total_victims = Male_victims + Female_victims
print(f'The percentage of males is: {Male_victims / total_victims * 100:.2f}%, and for Females : {Female_victims / total_victims * 100:.2f}%')
print(f'{round(Male_victims / Female_victims)} males for every 1 Female')

In [ ]:
#top 25 most dangerous cities
data.groupby('city_or_county')['total_victims'].sum().sort_values(ascending=False).head(25)

In [ ]:
#The most gun type used in crimes
data['gun_type'].value_counts()